# Sales Assist Tool - Multi-Agent Workflow

This notebook demonstrates a complete agentic AI workflow for sales assistance from the seller's perspective.

**Workflow:** Seller Query → Supervisory Agent → Contract Agent → Research Agent → Action Agent → Recommended Actions

## Agents:
1. **Supervisory Agent** - Orchestrates the entire workflow and interprets seller intent
2. **Contract Agent** - Reads and analyzes ESA/contracts from file path (OCR + ingestion)
3. **Research Agent** - Enriches partner context with internal sales data and external signals via Tavily
4. **Action Agent** - Determines next best sales action and creates artifacts

**System Process:**
1. Supervisory agent interprets intent: Contract signed, determine next best sales action and assess risk
2. Contract Agent reads Confluent ESA from file path, performs OCR + ingestion, returns structured summary
3. Research Agent retrieves Confluent sales history and external context via Tavily MCP
4. Action Agent analyzes next best action, assesses risk level, pulls historical patterns, creates artifacts (CRM update, draft email)
5. IBM Seller receives: Next Best Step recommendation, risk assessment, draft follow-up email, CRM confirmation

**Note:** In this demo, contracts are read from file paths (not uploaded). The Research Agent uses Tavily for external web search.

## Setup

In [14]:
#import sys
#!{sys.executable} -m pip install python-dotenv --index-url https://pypi.org/simple

In [15]:
#!pip install --upgrade pip

In [3]:
from dotenv import load_dotenv
print("Success")

Success


In [4]:
# Install all requirements from requirements.txt
import sys
import subprocess

print("Installing packages from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("All packages installed successfully!\n")
except subprocess.CalledProcessError as e:
    print(f"Error installing packages: {e}\n")
except FileNotFoundError:
    print("requirements.txt file not found!\n")

# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")


Installing packages from requirements.txt...
All packages installed successfully!

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


In [5]:
# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


## Demo 1: Complete Workflow with Supervisory Agent

This demonstrates the full workflow orchestrated by the Supervisory Agent.

**Note:** The Research Agent uses Tavily for external web search to enrich partner context with real-time market intelligence.

In [6]:
from supervisory_agent import SupervisoryAgent

# Initialize the Supervisory Agent
# Note: The intent analysis output may include some LLM reasoning text
# This is normal - the important part is that all agents execute correctly
print("Initializing Supervisory Agent...")
supervisor = SupervisoryAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)
print("Supervisory Agent ready\n")

Initializing Supervisory Agent...
Loaded scenario actions from docs/ScenarioActions.pdf (5143 characters)
Supervisory Agent ready



In [7]:
# IBM Seller scenarios supported by this demo
workflow_examples = [
    "Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?",
    "Can you give me an overview of all of the current contracts coming up for renewal, any contracts that have expired recently. Recommended next steps to take?",
    "I'm going to reach out to the CPO can you draft me an email?"
]

# Choose one of the supported workflows
seller_query = workflow_examples[0]
partner_name = "Confluent"

print("="*80)
print("SALES ASSIST WORKFLOW - DEMO")
print("="*80)
print("Supported workflows:")
for idx, example in enumerate(workflow_examples, start=1):
    print(f"  {idx}. {example}")
print(f"\nSelected Seller Query: {seller_query}")
print("Contract Scope: all files in docs/ beginning with Confluent_IBM")
print(f"Partner: {partner_name}")
print("\n" + "="*80)
print("Starting workflow...")
print("="*80 + "\n")

SALES ASSIST WORKFLOW - DEMO
Supported workflows:
  1. Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?
  2. Can you give me an overview of all of the current contracts coming up for renewal, any contracts that have expired recently. Recommended next steps to take?
  3. I'm going to reach out to the CPO can you draft me an email?

Selected Seller Query: Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?
Contract Scope: all files in docs/ beginning with Confluent_IBM
Partner: Confluent

Starting workflow...



In [8]:
# Run the complete workflow across all Confluent_IBM contracts in docs/
result = supervisor.run(
    seller_query=seller_query,
    contract_file_path=None,
    partner_name=partner_name
)


SUPERVISORY AGENT - Workflow Initialization
Seller Query: Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"6c919e13a8127b6004098eee0fe53e29","status_code":429}



Intent Analysis:
RULE_BASED_FALLBACK

Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
DEBUG: Reading document from: docs/Confluent_IBM-1.30.2024.docx
DEBUG: Extracted 11653 characters
DEBUG: Returning raw_text with 11653 characters


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"926e1328f86fa57296ac3057f5a4cdf1","status_code":429}
Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"9aada8b4c39d74c84

DEBUG: Reading document from: docs/Confluent_IBM-1.30.2025.docx
DEBUG: Extracted 11820 characters
DEBUG: Returning raw_text with 11820 characters


...


Contract Agent completed successfully
Contracts processed: 4

EXECUTING RESEARCH AGENT
Researching: Confluent
DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Products', 'Next Steps', 'Agent Next Steps']
DEBUG - First few rows:
        Opportunity Name Owner Full Name Stage     Amount  \
0       Confluent Cognos    Kylie Brittz   Won  1000000.0   
1  Confluent watsonx ESA       Anand Das   Won   250000.0   

            Close Date                                           Products  \
0  2023-05-30 00:00:00                                             Cognos   
1  2025-01-30 00:00:00  watsonx Orchestrate, watsonx.governance, watso...   

                                          Next Steps  \
0  Review all active Confluent contracts and conf...   
1  Review all active Confluent contracts and conf...   

                                    Agent Next Steps  
0  Review all active Confluent contracts and conf...  
1  Review all active Co

Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"d4b7cacaeea8eb9f96d46ba611ad6bb7","status_code":429}



Research Agent error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"d4b7cacaeea8eb9f96d46ba611ad6bb7","status_code":429}

EXECUTING ACTION AGENT


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"eba364f625467ba3629605e23fde464c","status_code":429}



Action Agent error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"eba364f625467ba3629605e23fde464c","status_code":429}


In [9]:
# Display the final result
print("\n" + "="*80)
print("WORKFLOW COMPLETE - FINAL RESULT")
print("="*80 + "\n")
print(result["final_result"])


WORKFLOW COMPLETE - FINAL RESULT

SALES ASSIST TOOL - WORKFLOW COMPLETE

Original Query: Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?
Timestamp: 2026-04-07 11:11:10

EXECUTIVE SUMMARY

Partner: Confluent
Maturity Level: Unknown
Sales Velocity: Unknown

CONTRACT PORTFOLIO OVERVIEW

Total Contracts: 4
Renewal Candidates: 0
Recently Expired: 0

- Review the active IBM-Confluent contract portfolio and confirm which agreements are currently enabled for sales activity.
- Prioritize contracts with renewal dates or notice windows in the next 30-90 days.
- Update the CRM Agent Next Steps column with the highest-priority seller actions for Confluent.

WORKFLOW COMPLETE

Contract portfolio analyzed and ingested
Partner research completed
Next best action determined
Artifacts created (CRM update, draft email)

Seller can now execute the recommended action directly from this tool.


## Demo 2: Individual Agent Testing

Test each agent independently to understand their specific functions.

### 2.1 Contract Agent - Document Analysis

In [10]:
from contract_agent import ContractAgent

# Initialize Contract Agent
contract_agent = ContractAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

print("Testing Contract Agent...")
print("="*80)

# Process contract
contract_result = contract_agent.run("docs/Confluent_IBM-1.30.2024.docx")

print("\n" + "="*80)
print("CONTRACT AGENT OUTPUT")
print("="*80)
print(f"\nDocument Length: {len(contract_result.get('raw_text', ''))} characters")
print(f"Vector Store Status: {contract_result.get('vector_store_status', 'Unknown')}")

# Display structured summary if available
if contract_result.get('structured_summary'):
    import json
    print("\nStructured Summary:")
    print(json.dumps(contract_result['structured_summary'], indent=2))

Testing Contract Agent...
DEBUG: Reading document from: docs/Confluent_IBM-1.30.2024.docx
DEBUG: Extracted 11653 characters
DEBUG: Returning raw_text with 11653 characters


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"21758573e595bdf8450e34dea3fadc5f","status_code":429}
Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"a75c92952ce311fa0


CONTRACT AGENT OUTPUT

Document Length: 11653 characters
Vector Store Status: Failed - Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/embeddings?version=2026-03-18)
Status code: 403, body: {"errors":[{"code":"token_quota_reached","message":"Request of 1 token(s) from quota was rejected","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-embeddings"}],"trace":"6ea0e2905388ec9a6de8f4842ac118b4","status_code":403}

Structured Summary:
{
  "parties": [
    "Confluent",
    "IBM"
  ],
  "effective_date": "Jan 31, 2024",
  "term_length": "Not specified",
  "key_obligations": [
    "renewal review",
    "contract management"
  ],
  "milestones": [],
  "contract_type": "Sales Agreement",
  "risk_level": "Medium",
  "fallback_reason": "Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)\nStatus code: 429, body: {\"errors\":[{\"code\":\"consumption_limit_reached\",\"message\":\"The usage limit for the cu

### 2.2 Research Agent - Partner Intelligence

In [13]:
from research_agent import research_partner

print("Testing Research Agent...")
print("="*80)

# Research partner
partner_profile = research_partner("Confluent")

print("\n" + "="*80)
print("RESEARCH AGENT OUTPUT")
print("="*80)
print(f"\nPartner: {partner_profile.get('partner_name', 'Unknown')}")
print(f"Maturity Level: {partner_profile.get('maturity_level', 'Unknown')}")
print(f"Sales Velocity: {partner_profile.get('sales_velocity', 'Unknown')}")
print(f"\nDeal Blockers: {len(partner_profile.get('deal_blockers', []))}")

# Display synthesis (truncated)
synthesis = partner_profile.get('synthesis', '')
if synthesis:
    print("\nPartner Intelligence Summary:")
    print(synthesis[:500] + "..." if len(synthesis) > 500 else synthesis)

Testing Research Agent...
DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Products', 'Next Steps', 'Agent Next Steps']
DEBUG - First few rows:
        Opportunity Name Owner Full Name Stage     Amount  \
0       Confluent Cognos    Kylie Brittz   Won  1000000.0   
1  Confluent watsonx ESA       Anand Das   Won   250000.0   

            Close Date                                           Products  \
0  2023-05-30 00:00:00                                             Cognos   
1  2025-01-30 00:00:00  watsonx Orchestrate, watsonx.governance, watso...   

                                          Next Steps  \
0  Schedule a portfolio review with the Confluent...   
1  Schedule a portfolio review with the Confluent...   

                                    Agent Next Steps  
0  Schedule a portfolio review with the Confluent...  
1  Schedule a portfolio review with the Confluent...  
DEBUG - Final column names: ['Opportunity Name', 'Own

Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"beeaf3754f7e0d4982a4fd5206709e4c","status_code":429}


ApiRequestFailure: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2026-03-18)
Status code: 429, body: {"errors":[{"code":"consumption_limit_reached","message":"The usage limit for the current plan has been reached: the total number of free concurrent requests for model meta-llama/llama-3-3-70b-instruct has reached its limit 10. Please try again later","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"beeaf3754f7e0d4982a4fd5206709e4c","status_code":429}

### 2.3 Action Agent - Next Best Action

In [12]:
from action_agent import ActionAgent

# Initialize Action Agent
action_agent = ActionAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

print("Testing Action Agent...")
print("="*80)

# Run action agent with contract and partner data
action_result = action_agent.run(
    contract_summary=contract_result,
    partner_profile=partner_profile
)

print("\n" + "="*80)
print("ACTION AGENT OUTPUT")
print("="*80)

# Display risk assessment
risk = action_result.get('risk_assessment', {})
print(f"\nRisk Level: {risk.get('risk_level', 'Unknown')}")
print(f"Risk Score: {risk.get('risk_score', 0)}/100")

# Display recommended action
action = action_result.get('recommended_action', {})
if action:
    print("\nRecommended Action:")
    print(action.get('raw_recommendation', 'No recommendation')[:300] + "...")

# Display draft email (truncated)
email = action_result.get('draft_email', '')
if email:
    print("\nDraft Email (preview):")
    print(email[:300] + "..." if len(email) > 300 else email)

Loaded scenario actions from docs/ScenarioActions.pdf (5143 characters)
Testing Action Agent...


NameError: name 'partner_profile' is not defined

## Demo 3: Different Scenarios

Test the workflow with different contract scenarios.

In [ ]:
# IBM Seller scenarios - working with Confluent as partner/customer
scenarios = [
    {
        "query": """I'm preparing for a renewal discussion with Confluent. 
        Can you review their current contract and tell me what leverage points 
        I have based on their usage and our past relationship? I need to know 
        if I should push for expansion or focus on retention.""",
        "contract": "docs/Confluent_IBM-3.29.2024.docx",
        "partner": "Confluent"
    },
    {
        "query": """Confluent's contract is coming up for renewal in 60 days. 
        I need to understand their historical buying patterns and any red flags 
        from previous deals. What should my outreach strategy be?""",
        "contract": "docs/Confluent_IBM-5.30.2023.docx",
        "partner": "Confluent"
    },
    {
        "query": """I'm working on upselling Confluent to additional IBM products. 
        Can you analyze their current contract, check what they're already using, 
        and recommend which products would be the best fit based on their profile?""",
        "contract": "docs/Confluent_IBM-1.30.2024.docx",
        "partner": "Confluent"
    }
]

print("Testing Multiple Scenarios...")
print("="*80 + "\n")

for i, scenario in enumerate(scenarios, 1):
    print(f"\nScenario {i}: {scenario['query']}")
    print("-"*80)
    
    # Check if file exists
    if not os.path.exists(scenario['contract']):
        print(f"⚠️  Contract file not found: {scenario['contract']}")
        continue
    
    try:
        result = supervisor.run(
            seller_query=scenario['query'],
            contract_file_path=scenario['contract'],
            partner_name=scenario['partner']
        )
        
        # Display key results
        print("\nWorkflow completed successfully")
        
        # Extract key information
        risk = result.get('risk_assessment', {})
        print(f"  Risk Level: {risk.get('risk_level', 'Unknown')}")
        
        action = result.get('recommended_action', {})
        if action:
            rec = action.get('raw_recommendation', '')
            # Extract just the ACTION line if present
            for line in rec.split('\n'):
                if line.startswith('ACTION:'):
                    print(f"  {line}")
                    break
        
    except Exception as e:
        print(f"Error: {str(e)}")

print("\n" + "="*80)
print("All scenarios tested")
print("="*80)

Testing Multiple Scenarios...


Scenario 1: I'm preparing for a renewal discussion with Confluent. 
        Can you review their current contract and tell me what leverage points 
        I have based on their usage and our past relationship? I need to know 
        if I should push for expansion or focus on retention.
--------------------------------------------------------------------------------

SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm preparing for a renewal discussion with Confluent. 
        Can you review their current contract and tell me what leverage points 
        I have based on their usage and our past relationship? I need to know 
        if I should push for expansion or focus on retention.

Intent Analysis:
KEY_CONCERNS: [Any concerns or goals mentioned]
RECOMMENDATION: [Recommended course of action based on the query]

INTENT: 
PARTNER_NAME: 
WORKFLOW_TYPE: 
CONTRACT_MENTIONED: 
KEY_ENTITIES: 
KEY_CONCERNS: 
RECOMMENDATION: 
```


INTENT: Prepare

## Summary

This notebook demonstrates a complete multi-agent workflow for sales assistance:

### Key Features:
- **Supervisory Agent**: Orchestrates the entire workflow
- **Contract Agent**: Analyzes contracts using RAG (Retrieval Augmented Generation)
- **Research Agent**: Enriches context with internal CRM data and external web search
- **Action Agent**: Determines next best action using historical patterns

### Technologies Used:
- **LangGraph**: For agent orchestration and state management
- **Watsonx**: For LLM inference and embeddings
- **Chroma**: For vector database storage
- **Tavily**: For web search and external context

### Benefits:
1. **Automated Analysis**: Reduces manual contract review time
2. **Contextual Intelligence**: Combines internal and external data
3. **Actionable Recommendations**: Provides specific next steps
4. **Scalable**: Can handle multiple contracts and partners

### Next Steps:
- Integrate with actual CRM system
- Add more sophisticated risk models
- Implement feedback loops for continuous improvement
- Deploy as a production service